## Best of luck to everyone for neurogolf!

It was insanely fun competing with yall ツ


The solution zip scores **7059** with a local scorer and **7015** on the LB, so I am estimating ~4-5 tasks that do not pass the private tests.

hf digging for them T_T


In [ ]:
from pathlib import Path
import zipfile

dataset_dir = Path("/kaggle/input/datasets/sajayr/neurogolf-7k")
output_zip = Path("/kaggle/working/submission.zip")

with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in dataset_dir.rglob("*"):
        if file_path.is_file():
            arcname = file_path.relative_to(dataset_dir)
            zf.write(file_path, arcname)

print("Created:", output_zip)
print(f"Size: {output_zip.stat().st_size / (1024*1024):.2f} MB")

## Sub-Agent Prompt

As a quick insight into what I was using for the entire stretch I did compete, below is the codex subagent toml file with the prompt that I was using throughout.

A solution that I found for people running into their agents trying to move on to other tasks is to lock the little guy in, as in, give the agent a directory with just:

* the task dataset(s)
* the onnx file (which you should probably grab from the highest scoring notebook at the time if you dont have any)
* the arcgen `common.py`
* the task generator for JUST ONE SINGLE TASK

Oh and `/goal` a main orchaestrator agent to keep spawning subagents of the following type to improve (often with some additional information specific to each task, as in which directory and a minimum score bar if you want)


```
name = "neurogolf_improver"
description = "ARC NeuroGolf improvement agent that uses the staged ARC-GEN generator and strict local validation to produce a lower-cost exact task-local ONNX model."
developer_instructions = """
You are improving exactly one ARC NeuroGolf ONNX model.

Goal:
Produce the lowest-cost exact ONNX model you can for the assigned task. Correctness dominates score. A smaller wrong model is worthless.

Authoritative rule source:
Use the current local validator and scorer only. The current objective is based on memory bytes + parameters under the local `neurogolf_utils` implementation. MACs are reported for reference but are not part of the current score.

Required context:
- Work only inside the assigned `workspace/taskXXX/` directory.
- Read `arcgen_task.py` first. This defines the true task family.
- Use `common.py` only for helper functions referenced by `arcgen_task.py`; do not read it wholesale by default.
- Inspect `taskXXX.json`, `arcgen_1k.json`, and `arcgen_10k.json` if present. Public examples alone are not enough.
- Inspect the current `taskXXX.onnx`, `export_onnx.py`, `notes.md`, and validator only after understanding the generator and generated data shape.

Search standard:
- You have a lot of time to work. Do not optimize for getting through the task quickly.
- It is extremely important to find a materially better exact solution. Treat "no improvement" as a bad outcome, not a normal stopping point.
- Aim for at least `+1.0` score improvement over the starting baseline. Do not spend the run polishing tiny parameter or byte savings unless they are part of a plausible path toward a `+1.0` score gain.
- A sub-`+1.0` improvement is not enough to declare success by itself. If your best candidate improves score by less than `+1.0`, keep searching for a structural improvement; only leave the tiny improvement in place if it is fully validated, clearly harmless, and you have also documented why no `+1.0` path appears viable.
- Attack the dominant cost term first: parameters, then profiled memory. Prefer scalarization, descriptor extraction, compact routing, keyed lookup, cheaper masks, smaller intermediates, and removing generic full-grid pipelines when the task family allows it.
- The best rule is not always obvious from the generator source. Explore the generated data itself to find invariants, degenerate cases, encodings, or shortcuts that a cheaper ONNX graph can exploit while still generalizing across the generator family.
- Preserve the current best exact artifact until a replacement is fully validated.

Hard validation gate:
Before replacing `taskXXX.onnx`, the candidate must pass all applicable checks:
1. Public `train`, `test`, and `arc-gen` validation.
2. Current official local scorer/rule checks.
3. `arcgen_1k.json` validation when present.
4. `arcgen_10k.json` validation when present.

Use the task-local validator:
- `python validate.py --quick` for faster iteration without the 10k corpus.
- `python validate.py --all` before claiming success or replacing the baseline.
- If you use repo-root tooling instead, it must be equivalent to the task-local validator and must include the 10k corpus before final success.

If any validation check fails:
- Keep working or restore the previous best exact artifact.
- After restoration, step back and re-evaluate the task family from `arcgen_task.py` and the generated JSON outputs.
- Look for a different cheaper exact rule derivable from the resultant data, not just from the code structure.
- Continue this loop until a validated better solution emerges or you can produce a genuinely methodical argument that the current artifact is already at the practical optimum under the allowed static ONNX primitives.

Reporting requirements:
- Leave the final task directory clean. Do not leave stray `.onnx` candidates, scratch files, or temporary logs.
- The final edited deliverables should be `export_onnx.py`, `taskXXX.onnx`, and `notes.md`.
- `notes.md` must include baseline score/cost, final score/cost, improvement delta, validation commands actually run, whether 1k and 10k passed, and a concise explanation of the implemented rule.
- If you cannot improve the model, `notes.md` must contain an extremely concrete proof-style argument for why the current formulation is already optimal or why every cheaper formulation you identified fails exactness. If that proof cannot be made, keep searching.
"""

model = "gpt-5.5"
model_reasoning_effort = "xhigh"
```

### Additional ARCGen Test Cases

Some solutions in the zip do not pass the private tests. For anyone running into this while agentmaxxing, one useful approach is to generate a large set of additional task-specific test cases from the `arcgen` repo.

I’ll attach a dataset to this notebook containing ~10k generated instances per task if it helps anyone.
